# InputOptions And OutputOptions

`pyneat.InputOptions` describes the contract at a graph input boundary: what kind of payload the application will push, what shape/format it has, and how the `appsrc` side should behave.

`pyneat.OutputOptions` describes the contract at one graph output endpoint: how many produced samples can queue, whether old samples are dropped, whether output is clock-synchronized, and how named multi-producer outputs combine samples.

This notebook uses a tiny pass-through graph so the boundary behavior is visible without a model archive.

## Imports

Run this notebook from the DevKit pyneat environment.

In [1]:
import numpy as np
import pyneat


## Mental Model

A push-mode graph has two public boundaries:

```text
application tensor/sample
  -> Input node configured by InputOptions
  -> graph stages
  -> Output node configured by OutputOptions
  -> application pull()
```

`InputOptions` answers: what will the app push? `OutputOptions` answers: how should produced samples wait for the app to pull them?

## Complete InputOptions Field Map

| Field | Default | What it controls |
| --- | --- | --- |
| `payload_type` | `PayloadType.Auto` | Semantic payload family: image, tensor, encoded, or infer from sample/tensor metadata. |
| `format` | empty | Pixel/tensor/encoded subformat, for example `RGB`, `BGR`, `NV12`, `FP32`, or `H264`. |
| `width` | `-1` | Input width; `-1` leaves unspecified. |
| `height` | `-1` | Input height; `-1` leaves unspecified. |
| `depth` | `-1` | Channel/tensor depth; `-1` leaves unspecified. |
| `max_width` | `-1` | Push-time validation maximum; does not negotiate caps. |
| `max_height` | `-1` | Push-time validation maximum; does not negotiate caps. |
| `max_depth` | `-1` | Push-time validation maximum; does not negotiate caps. |
| `fps_n` | `0` | Caps framerate numerator; `0/1` means unspecified. |
| `fps_d` | `1` | Caps framerate denominator. |
| `caps_override` | empty | Full caps string override for special cases such as multi-tensor caps. |
| `is_live` | `True` | Sets appsrc live behavior. Use true for cameras/live streams. |
| `do_timestamp` | `True` | Let appsrc timestamp pushed buffers. |
| `block` | `True` | Block on appsrc/pool pressure instead of dropping at that boundary. |
| `stream_type` | `0` | Appsrc stream type: `0` stream, `1` seekable, `2` random-access. |
| `max_bytes` | `0` | Appsrc back-pressure threshold; `0` means unlimited. |
| `use_simaai_pool` | `True` | Allocate from the SiMa-aware buffer pool. |
| `pool_min_buffers` | `1` | Minimum buffers held by the input pool. |
| `pool_max_buffers` | `2` | Maximum buffers held by the input pool. |
| `memory_policy` | `InputMemoryPolicy.Auto` | Ingress allocation target: auto, EV74, DMS0, or system memory. |
| `buffer_name` | empty | Optional buffer-name hint for legacy/interoperability paths. |
| `preprocess_meta` | unset | Optional metadata template for preprocess geometry and box-decode coordinate mapping. |

In [2]:
def public_names(obj):
    return [name for name in dir(obj) if not name.startswith("_")]

input_defaults = pyneat.InputOptions()
print("InputOptions fields:")
print(public_names(input_defaults))
print()
print("Defaults:")
for name in [
    "payload_type", "format", "width", "height", "depth",
    "max_width", "max_height", "max_depth", "fps_n", "fps_d",
    "is_live", "do_timestamp", "block", "max_bytes",
    "use_simaai_pool", "pool_min_buffers", "pool_max_buffers", "memory_policy",
]:
    print(name, getattr(input_defaults, name))


InputOptions fields:
['block', 'buffer_name', 'caps_override', 'depth', 'do_timestamp', 'format', 'fps_d', 'fps_n', 'height', 'is_live', 'max_bytes', 'max_depth', 'max_height', 'max_width', 'memory_policy', 'payload_type', 'pool_max_buffers', 'pool_min_buffers', 'preprocess_meta', 'stream_type', 'use_simaai_pool', 'width']

Defaults:
payload_type PayloadType.Auto
format Format.Auto
width -1
height -1
depth -1
max_width -1
max_height -1
max_depth -1
fps_n 0
fps_d 1
is_live True
do_timestamp True
block True
max_bytes 0
use_simaai_pool True
pool_min_buffers 1
pool_max_buffers 2
memory_policy InputMemoryPolicy.Auto


## Payload Type

`payload_type` is the public semantic family of the data crossing the input boundary.

| Value | Meaning | Internal media family |
| --- | --- | --- |
| `Auto` | Infer from pushed tensor/sample metadata when possible. | inferred |
| `Image` | Decoded image pixels such as RGB, BGR, NV12, I420. | `video/x-raw` |
| `Tensor` | Already-shaped tensor payload. | `application/vnd.simaai.tensor` |
| `Encoded` | Encoded byte stream such as H.264. | `video/x-h264` |

For tutorials, set `payload_type` explicitly when it clarifies the contract.

In [3]:
print("PayloadType:", public_names(pyneat.PayloadType))
print("InputMemoryPolicy:", public_names(pyneat.InputMemoryPolicy))
print("CombinePolicy:", public_names(pyneat.CombinePolicy))


PayloadType: ['Auto', 'Encoded', 'Image', 'Tensor']
InputMemoryPolicy: ['Auto', 'Dms0', 'Ev74', 'SystemMemory']
CombinePolicy: ['ByFrame', 'ByPts', 'None_']


## Image Input Contract

Use this when the application pushes decoded image frames. For RGB/BGR images, `depth = 3`. For NV12/I420, keep the pixel format explicit and use the width/height of the visible frame.

In [4]:
def make_rgb_input_options(width: int, height: int, fps: int = 30):
    opt = pyneat.InputOptions()
    opt.payload_type = pyneat.PayloadType.Image
    opt.format = pyneat.Format.RGB
    opt.width = width
    opt.height = height
    opt.depth = 3
    opt.fps_n = fps
    opt.fps_d = 1
    opt.is_live = False
    opt.do_timestamp = True
    opt.use_simaai_pool = False
    return opt

rgb_input = make_rgb_input_options(160, 120)
print("format:", rgb_input.format)
print("shape:", rgb_input.width, rgb_input.height, rgb_input.depth)


format: Format.RGB
shape: 160 120 3


## NV12 Video Input Contract

NV12 is common for video decode, video sender, and model pipelines that should avoid unnecessary RGB/BGR conversion. The visible frame is still `width x height`; the backing payload has Y and UV planes.

In [5]:
def make_nv12_input_options(width: int, height: int, fps: int = 30):
    opt = pyneat.InputOptions()
    opt.payload_type = pyneat.PayloadType.Image
    opt.format = pyneat.Format.NV12
    opt.width = width
    opt.height = height
    opt.depth = 1
    opt.max_width = width
    opt.max_height = height
    opt.max_depth = 1
    opt.fps_n = max(1, fps)
    opt.fps_d = 1
    opt.is_live = True
    opt.caps_override = f"video/x-raw,format=NV12,width={width},height={height},framerate={max(1, fps)}/1"
    return opt

nv12_input = make_nv12_input_options(1280, 720, 30)
print("caps_override:", nv12_input.caps_override)


caps_override: video/x-raw,format=NV12,width=1280,height=720,framerate=30/1


## `caps_override`

`caps_override` is the escape hatch for `InputOptions`. Normally Neat builds the input caps string from structured fields such as `payload_type`, `format`, `width`, `height`, `depth`, and `fps_n`/`fps_d`.

For example, structured RGB fields can generate caps like:

```text
video/x-raw,format=RGB,width=640,height=480,depth=3,framerate=30/1
```

When `caps_override` is set, core uses that string first instead of generating caps from the structured fields. Use it when the structured fields cannot express the exact GStreamer caps a downstream stage expects, or when a tutorial/demo should make negotiation fully deterministic.

Common reasons to use it:

- A downstream plugin expects a specific raw-video caps string.
- The caps need memory tags, encoded-video details, or custom fields not represented by `InputOptions`.
- The input is a special case such as multi-tensor/custom caps.
- You want the notebook to advertise exactly `video/x-raw,format=NV12,width=...,height=...,framerate=...`.

Prefer structured fields for normal RGB/BGR/tensor inputs. Use `caps_override` only when the exact caps string matters.


## Tensor Input Contract

Use tensor input when the application already prepared data in the model's expected tensor shape and dtype. This path is different from decoded image input: the payload family is tensor, not image.

In [6]:
def make_fp32_tensor_input_options(width: int, height: int, depth: int):
    opt = pyneat.InputOptions()
    opt.payload_type = pyneat.PayloadType.Tensor
    opt.format = pyneat.Format.FP32
    opt.width = width
    opt.height = height
    opt.depth = depth
    opt.is_live = False
    opt.use_simaai_pool = False
    return opt

tensor_input = make_fp32_tensor_input_options(224, 224, 3)
print("tensor input:", tensor_input.payload_type, tensor_input.format, tensor_input.width, tensor_input.height, tensor_input.depth)


tensor input: PayloadType.Tensor Format.FP32 224 224 3


## Caps, Pool, And Back-Pressure

`InputOptions` becomes an `appsrc` boundary. These fields affect how that boundary behaves:

| Field | Practical meaning |
| --- | --- |
| `caps_override` | Replace generated caps with a full caps string when generated fields are not enough. |
| `is_live` | Treat pushed data as live; useful for camera/video timing. |
| `do_timestamp` | Let appsrc assign timestamps when the app does not provide them. |
| `block` | Let appsrc block under pressure. This is separate from `RunOptions.overflow_policy`. |
| `max_bytes` | Back-pressure threshold at appsrc; `0` means unlimited. |
| `use_simaai_pool` | Use SiMa-aware allocation. Disable for simple CPU/notebook examples. |
| `memory_policy` | Force allocation target only when the pipeline needs it. |

## Preprocess Metadata

`preprocess_meta` is an advanced field. It can attach geometry and affine metadata to ingress buffers so downstream stages, such as box decode, can map model outputs back to source image coordinates.

Most tutorials should leave it unset. Prefer `ModelOptions.preprocess` for normal model-managed preprocessing.

In [7]:
meta = pyneat.PreprocessMetaTemplate()
print("PreprocessMetaTemplate fields:")
print(public_names(meta))
print("enabled:", meta.enabled)


PreprocessMetaTemplate fields:
['axis_perm', 'color_in', 'color_out', 'enabled', 'normalize', 'pad_value', 'quantize', 'resize_mode', 'roi_input_batch_size', 'roi_list_enabled', 'roi_pad_value', 'roi_source_height', 'roi_source_stride_bytes', 'roi_source_width', 'rois', 'scaled_height', 'scaled_width', 'target_height', 'target_width', 'tessellate']
enabled: False


## Complete OutputOptions Field Map

| Field | Default | What it controls |
| --- | --- | --- |
| `max_buffers` | `4` | Maximum queued samples at the output endpoint; `0` means unbounded. |
| `drop` | `False` | Drop old queued samples on overflow instead of applying back-pressure. |
| `sync` | `False` | Sync output to the pipeline clock. Useful for clocked/realtime playback-style outputs. |
| `combine_policy` | `CombinePolicy.None_` | How a named output combines multiple upstream producers. |

In [8]:
output_defaults = pyneat.OutputOptions()
print("OutputOptions fields:")
print(public_names(output_defaults))
print()
print("Defaults:")
print("max_buffers:", output_defaults.max_buffers)
print("drop:", output_defaults.drop)
print("sync:", output_defaults.sync)
print("combine_policy:", output_defaults.combine_policy)


OutputOptions fields:
['clocked', 'combine_policy', 'drop', 'every_frame', 'latest', 'max_buffers', 'sync']

Defaults:
max_buffers: 4
drop: False
sync: False
combine_policy: CombinePolicy.None_


## OutputOptions Presets

Prefer the preset constructors for common cases.

| Preset | Meaning | Typical use |
| --- | --- | --- |
| `OutputOptions.every_frame(max_buffers=30)` | Keep a bounded queue and do not drop. | Debugging, correctness, frame-by-frame pulls. |
| `OutputOptions.latest()` | Preset intended for freshness-oriented output. | Low-latency live preview/analytics. |
| `OutputOptions.clocked(max_buffers=1)` | Drop when needed and sync to pipeline clock. | Realtime-paced output endpoints. |

In [9]:
def summarize_output_options(name, opt):
    print(name, "max_buffers=", opt.max_buffers, "drop=", opt.drop, "sync=", opt.sync, "combine=", opt.combine_policy)

summarize_output_options("default", pyneat.OutputOptions())
summarize_output_options("every_frame", pyneat.OutputOptions.every_frame(4))
summarize_output_options("latest", pyneat.OutputOptions.latest())  # Inspect actual installed preset fields.
summarize_output_options("clocked", pyneat.OutputOptions.clocked(1))


default max_buffers= 4 drop= False sync= False combine= CombinePolicy.None_
every_frame max_buffers= 4 drop= False sync= False combine= CombinePolicy.None_
latest max_buffers= 4 drop= False sync= False combine= CombinePolicy.None_
clocked max_buffers= 1 drop= True sync= True combine= CombinePolicy.None_


## Build A Pass-Through Graph

This graph uses `InputOptions` on the input side and `OutputOptions` on the output side. There are no processing stages between them.

In [10]:
HEIGHT = 120
WIDTH = 160

image_rgb = np.zeros((HEIGHT, WIDTH, 3), dtype=np.uint8)
image_rgb[..., 0] = 40
image_rgb[..., 1] = 100
image_rgb[..., 2] = 180
tensor = pyneat.Tensor.from_numpy(image_rgb, copy=True, image_format=pyneat.PixelFormat.RGB)

input_options = make_rgb_input_options(WIDTH, HEIGHT, fps=30)
output_options = pyneat.OutputOptions.every_frame(4)

graph = pyneat.Graph("input_output_options_pass_through")
graph.add(pyneat.nodes.input(input_options))
graph.add(pyneat.nodes.output("image", output_options))

print(graph.describe())


0) Input  [mysrc]
1) Output  [image]


## Run The Graph

The input contract validates what we push. The output endpoint buffers the produced sample until the application pulls it.

In [11]:
run_options = pyneat.RunOptions()
run_options.output_memory = pyneat.OutputMemory.Owned

run = graph.build([tensor], run_options)
try:
    if not run.push([tensor]):
        raise RuntimeError(f"push failed: {run.last_error()}")
    sample = run.pull("image", 1000)
    if sample is None:
        raise RuntimeError("timed out waiting for output")
    tensors = list(getattr(sample, "tensors", []))
    if not tensors and getattr(sample, "tensor", None) is not None:
        tensors = [sample.tensor]
    out = tensors[0].to_numpy(copy=True)
    print("output shape:", out.shape)
    print("same pixels:", bool(np.array_equal(out, image_rgb)))
finally:
    run.close()


[1/4] Initializing runtime...
[2/4] Preparing input stream...
[3/4] Building graph...


output shape: (120, 160, 3)
same pixels: True


[4/4] Graph ready (6 ms)


## Named Boundaries

`pyneat.nodes.input("name", options)` creates a named ingress endpoint for `run.push("name", ...)`. `pyneat.nodes.output("name", options)` creates a named egress endpoint for `run.pull("name", ...)`.

Use names when a graph has multiple inputs or outputs.

In [12]:
named_graph = pyneat.Graph("named_boundaries")
named_graph.add(pyneat.nodes.input("camera", input_options))
named_graph.add(pyneat.nodes.output("preview", pyneat.OutputOptions.latest()))

print(named_graph.describe())


0) Input  [camera]
1) Output  [preview]


## CombinePolicy

`combine_policy` is only meaningful for named public graph outputs with more than one upstream producer.

| Value | Meaning |
| --- | --- |
| `None_` | Do not combine; multiple producers are rejected. |
| `ByFrame` | Combine one sample from each producer when `frame_id` matches. |
| `ByPts` | Combine one sample from each producer when presentation timestamp `pts_ns` matches. |

For ordinary single-output graphs, leave it at `None_`.

In [13]:
combined_output = pyneat.OutputOptions.every_frame(4)
combined_output.combine_policy = pyneat.CombinePolicy.ByFrame
summarize_output_options("combined_by_frame", combined_output)


combined_by_frame max_buffers= 4 drop= False sync= False combine= CombinePolicy.ByFrame


## InputOptions vs OutputOptions vs RunOptions

| Object | Applied at | Controls |
| --- | --- | --- |
| `InputOptions` | `pyneat.nodes.input(...)` | Input payload contract, caps, appsrc behavior, input buffer pool. |
| `OutputOptions` | `pyneat.nodes.output(...)` | One output endpoint's appsink queue, drop, sync, combine behavior. |
| `RunOptions` | `graph.build(...)` / `graph.run(...)` | Whole-runtime queueing, overflow, output memory ownership, preflight, diagnostics. |

Do not use `OutputOptions` when you mean whole-runtime overflow policy. That belongs to `RunOptions`.

## Decision Guide

| Goal | InputOptions | OutputOptions |
| --- | --- | --- |
| CPU notebook RGB image | `PayloadType.Image`, `Format.RGB`, `use_simaai_pool=False` | `every_frame(4)` |
| Live RTSP decoded NV12 frame | `PayloadType.Image`, `Format.NV12`, explicit width/height/fps | `latest()` or `every_frame(1)` |
| Preprocessed tensor input | `PayloadType.Tensor`, tensor dtype/shape | `every_frame(...)` |
| Debug every output frame | clear input caps | `every_frame(max_buffers)` |
| Lowest-latency preview | clear input caps | `latest()` |
| Clock-paced output | clear input caps | `clocked(max_buffers)` |

## Common Mistakes

- Leaving image shape unspecified when the downstream stage expects fixed caps.
- Using RGB/BGR caps for NV12 payloads or the reverse.
- Treating `max_width` and `max_height` as negotiated caps; they are push-time limits.
- Using `OutputOptions.latest()` for correctness tests that need every frame.
- Expecting `OutputOptions.drop` to control input-side queue overflow; use `RunOptions.overflow_policy` for runtime input overflow.
- Keeping `use_simaai_pool=True` in simple CPU-only notebook examples when system memory is easier to inspect.

## What To Remember

- `InputOptions` defines what the application is allowed to push into a graph.
- `OutputOptions` defines how one graph output endpoint buffers samples for pull.
- Set `payload_type`, `format`, `width`, `height`, and `depth` explicitly in tutorials.
- Use `caps_override` only when the structured fields cannot express the caps you need.
- Use `every_frame` for correctness/debugging and `latest` for low-latency live output.
- Named input/output nodes make push/pull code clearer in multi-boundary graphs.

## References

- Core input API: [Input.h](https://github.com/sima-neat/core/blob/main/include/nodes/io/Input.h)
- Core output API: [Output.h](https://github.com/sima-neat/core/blob/main/include/nodes/common/Output.h)
- Core graph tutorial: [build_inference_pipeline.py](https://github.com/sima-neat/core/blob/main/tutorials/004_build_inference_pipeline/build_inference_pipeline.py)
- Custom data graph tutorial: [build_a_custom_data_graph.py](https://github.com/sima-neat/core/blob/main/tutorials/013_build_a_custom_data_graph/build_a_custom_data_graph.py)
- Core tutorials: [https://github.com/sima-neat/core/tree/main/tutorials](https://github.com/sima-neat/core/tree/main/tutorials)
